# Parrot on Chameleon

This notebook is a Chameleon/Trovi-oriented reproduction scaffold for **Parrot: Efficient Serving of LLM-based Applications with Semantic Variable**. It covers **resource reservation, server provisioning, Parrot installation, smoke testing, and a Parrot-specific shared-prefix fill benchmark**.

## What this notebook does

- reserves a suitable Chameleon node and a floating IP
- launches a server image on the reserved hardware
- connects to the server over SSH from the notebook
- installs Parrot and its dependencies
- records machine and software metadata for artifact reporting
- provides smoke tests for the Parrot Python package and service stack
- runs a Parrot-specific shared-prefix fill benchmark and plots the results

## Notes

Parrot's public installation guide expects Linux, CUDA 12.1, PyTorch 2.1, and a GPU with compute capability >= 7.0. The paper reports experiments on A100 and A6000 GPUs. In practice, you should pick a GPU node manually based on current availability and how closely you want to match the paper.


## Reproduction Plan

The notebook follows this order:

1. configure Chameleon context
2. inspect available hardware and choose a target node type
3. create a lease with a floating IP
4. launch an Ubuntu server on the reserved node
5. attach the floating IP and verify connectivity
6. install Parrot, its dependencies, and optional benchmark dependencies
7. record environment metadata useful for artifact packaging
8. run smoke tests and benchmark entry points

If any step fails, stop there and capture the failure in the artifact notes rather than pushing forward blindly.


In [ ]:
from chi import context

# Opt into the newer notebook-friendly python-chi interface.
context.version = "1.0"

context.choose_site(default="CHI@TACC")
context.choose_project()


## Hardware Discovery

Parrot's paper used GPUs, so start by looking for GPU-capable nodes at your current site. Use the next cells to inspect what is available, then choose the GPU node type you want manually before creating the lease.


In [ ]:
from chi import hardware

# Display all nodes first if you want a broad overview.
# hardware.show_nodes()

# Common artifact workflow: filter for GPU nodes and inspect the table.
gpu_nodes = hardware.get_nodes(gpu=True)
print(f"Found {len(gpu_nodes)} GPU-capable nodes at the current site.")
hardware.show_nodes(gpu_nodes)


In [ ]:
# Inspect likely single-GPU candidates, then choose `node_type` manually below.
# The automatic preference order was not fully reliable in practice.
preferred_types = ["compute_liqid", "compute_gigaio", "gpu_h100"]
minimum_hours = 3

candidate_matches = {}
for candidate in preferred_types:
    matches = hardware.get_nodes(node_type=candidate, filter_reserved=True)
    candidate_matches[candidate] = matches
    print(f"\n=== {candidate}: {len(matches)} available nodes ===")
    if matches:
        hardware.show_nodes(matches)
        start, end = matches[0].next_free_timeslot(minimum_hours=minimum_hours)
        print(f"Example free slot: {start} -> {end}")

# Pick the GPU type you want after reviewing the tables above.
node_type = "compute_liqid"
print(f"Selected node_type: {node_type}")


## Reservation Parameters

The defaults below aim at a single-node run with one floating IP. Increase the duration if you expect long dependency builds or model downloads. For a multi-node or multi-GPU reproduction, adapt the reservation plan before submitting.

Important: choosing a GPU node type only gives you GPU hardware. You still need a GPU-capable image. For this notebook, use a CUDA image so that `nvidia-smi`, `nvcc`, and CUDA userspace are already present.


In [ ]:
from datetime import timedelta
import os

lease_name = f"{os.getenv('USER', 'user')}-parrot"
lease_hours = 8
image_name = "CC-Ubuntu22.04-CUDA"
reserve_floating_ip = True

print({
    "lease_name": lease_name,
    "lease_hours": lease_hours,
    "image_name": image_name,
    "reserve_floating_ip": reserve_floating_ip,
})


In [ ]:
from chi import lease

if node_type is None:
    raise RuntimeError("Set node_type before creating the lease.")

my_lease = lease.Lease(
    name=lease_name,
    duration=timedelta(hours=lease_hours),
)
my_lease.add_node_reservation(amount=1, node_type=node_type)
if reserve_floating_ip:
    my_lease.add_fip_reservation(amount=1)

my_lease.submit(wait_for_active=True)
print(f"Lease status: {my_lease.status}")
print(my_lease)


## Launch a Server on the Reserved Hardware

This creates the bare-metal instance from the Chameleon image catalog. Launch times can vary substantially depending on node type and site load.


In [ ]:
from chi import server

reservation_id = my_lease.node_reservations[0]["id"]
my_server = server.Server(
    name=lease_name,
    reservation_id=reservation_id,
    image_name=image_name,
)
my_server.submit(wait_for_active=True)
print(f"Server status: {my_server.status}")
print(my_server)


## Associate a Floating IP

If the lease reserved a floating IP, attach it here. If `get_reserved_floating_ips()` returns an empty list, pause and allocate/associate an IP through the Chameleon dashboard or recreate the lease with `add_fip_reservation(amount=1)`.


In [ ]:
floating_ips = my_lease.get_reserved_floating_ips()
print("Reserved floating IPs:", floating_ips)

if not floating_ips:
    raise RuntimeError(
        "No reserved floating IPs found. Check the lease contents, wait for the lease to be ACTIVE, "
        "or allocate an IP manually from the Chameleon dashboard."
    )

floating_ip = floating_ips[0]
my_server.associate_floating_ip(floating_ip)
my_server.check_connectivity(host=floating_ip)
print(f"Connected via floating IP: {floating_ip}")


## System Preparation

Parrot's public install instructions assume CUDA 12.1, PyTorch 2.1, and editable installs of both the top-level package and the bundled `3rdparty/vllm` tree. The commands below stage a virtual environment and clone the Parrot repo recursively.


In [ ]:
bootstrap_script = r'''
set -euxo pipefail

export DEBIAN_FRONTEND=noninteractive
sudo apt-get update
sudo apt-get install -y \
    git git-lfs build-essential pkg-config cmake ninja-build \
    python3-dev python3-venv python3-pip \
    curl wget htop jq

cd /home/cc
rm -rf ParrotServe
git clone https://github.com/microsoft/ParrotServe.git
cd /home/cc/ParrotServe

# Fix submodule cloning over HTTPS instead of SSH.
git config --global url."https://github.com/".insteadOf git@github.com:
git submodule update --init --recursive

python3 -m venv /home/cc/parrot-venv
source /home/cc/parrot-venv/bin/activate
python -m pip install --upgrade pip setuptools wheel

# The CUDA image exposes the toolkit under /usr/local/cuda-12.6 on current Chameleon builds.
export CUDA_HOME=/usr/local/cuda-12.6
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:${LD_LIBRARY_PATH:-}
echo "CUDA_HOME=$CUDA_HOME"
nvidia-smi
which nvcc
nvcc -V
python --version

# Mirror the published install guidance as closely as possible.
python -m pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu121
python -m pip install -r requirements.txt
# Editable builds in this repo currently need NumPy 1.x and pkg_resources from setuptools<81.
python -m pip install 'numpy<2' 'setuptools<81'
python -m pip install --force-reinstall 'setuptools<81'

cd /home/cc/ParrotServe/3rdparty/vllm
python -m pip install -e . --no-build-isolation || true

cd /home/cc/ParrotServe
python -m pip install -e . --no-build-isolation

# Optional benchmark extras from the public install note.
if [ -d /home/cc/ParrotServe/3rdparty/FastChat ]; then
  cd /home/cc/ParrotServe/3rdparty/FastChat
  python -m pip install -e ".[model_worker,webui]" || true
fi
if [ -d /home/cc/ParrotServe/3rdparty/langchain/libs/langchain ]; then
  cd /home/cc/ParrotServe/3rdparty/langchain/libs/langchain
  python -m pip install -e . || true
fi

cd /home/cc/ParrotServe
python - <<"PY"
import torch
import parrot
print("Torch version:", torch.__version__)
print("Imported parrot successfully:", getattr(parrot, "__file__", "unknown"))
PY
'''

with open("bootstrap.sh", "w") as f:
    f.write(bootstrap_script)

with my_server.ssh_connection() as conn:
    conn.put("bootstrap.sh", "/tmp/bootstrap.sh")
    result = conn.run(
        "bash /tmp/bootstrap.sh",
        timeout=7200,
        warn=True,
        pty=True,
    )

print(result.stdout)
print(result.stderr)
print(result.exited)


## Environment Variables and Credentials

If the benchmark path you choose requires model downloads from Hugging Face or other registries, populate the relevant environment variables here before continuing. Avoid hard-coding secrets into the notebook if it will be shared through Trovi.


In [ ]:
# Fill these in only if needed for your chosen model workflow.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
MODEL_NAME = os.environ.get("PARROT_MODEL_NAME", "")

remote_env_lines = []
if HF_TOKEN:
    remote_env_lines.append(f"export HF_TOKEN={HF_TOKEN!r}")
if MODEL_NAME:
    remote_env_lines.append(f"export PARROT_MODEL_NAME={MODEL_NAME!r}")

if remote_env_lines:
    remote_env_script = "\n".join(remote_env_lines) + "\n"
    with open("remote_env.sh", "w") as f:
        f.write(remote_env_script)
    with my_server.ssh_connection() as conn:
        conn.put("remote_env.sh", "/home/cc/parrot-env.sh")
        result = conn.run("cat /home/cc/parrot-env.sh", warn=True, pty=True)
    print(result.stdout)
else:
    print("No extra remote environment variables were set from the notebook environment.")


## Record the Repository Layout

This is a lightweight sanity check before trying to start services or benchmarks. It also helps document which entry points are actually present in the checked-out revision.


In [ ]:
repo_inspect_cmd = (
    "source /home/cc/parrot-venv/bin/activate; "
    "cd /home/cc/ParrotServe; "
    "pwd; "
    "ls; "
    "find docs -maxdepth 2 -type f | sort | head -n 50; "
    "find examples -maxdepth 2 -type f | sort | head -n 50 || true; "
    "find benchmark -maxdepth 3 -type f | sort | head -n 100 || true; "
    "find sample_configs -maxdepth 3 -type f | sort | head -n 100 || true"
)

with my_server.ssh_connection() as conn:
    result = conn.run(repo_inspect_cmd, timeout=1800, warn=True, pty=True)

print(result.stdout)
print(result.stderr)
print(result.exited)


## Python-Level Smoke Test

Start with the simplest possible check: can Python import Parrot and evaluate a tiny semantic function example on the frontend side? This validates the install before we involve the heavier serving stack.


In [ ]:
smoke_py = '''from parrot import P

@P.semantic_function()
def tell_me_a_joke(topic: P.Input, joke: P.Output):
    """Tell me a joke about {{topic}}: {{joke}}."""

print("Parrot frontend import and decorator smoke test passed.")
'''

with open("smoke_test.py", "w") as f:
    f.write(smoke_py)

with my_server.ssh_connection() as conn:
    conn.put("smoke_test.py", "/tmp/smoke_test.py")
    result = conn.run(
        "source /home/cc/parrot-venv/bin/activate && cd /home/cc/ParrotServe && python /tmp/smoke_test.py",
        timeout=1800,
        warn=True,
        pty=True,
    )

print(result.stdout)
print(result.stderr)
print(result.exited)


## Service Bring-Up Placeholder

The exact service-launch command may depend on which engine/backend config in the checked-out repo is the best match for your hardware. Use this cell to discover likely CLI entry points before committing to a long run.


In [ ]:
service_discovery_cmd = (
    "source /home/cc/parrot-venv/bin/activate && "
    "cd /home/cc/ParrotServe && "
    "pip show parrot && "
    "python -c \"import pkgutil, parrot; print('parrot package root:', parrot.__file__); print('top-level submodules:'); [print(' -', m.name) for m in pkgutil.iter_modules(parrot.__path__)]\" && "
    "find parrot -maxdepth 3 -type f | sort | head -n 200"
)

with my_server.ssh_connection() as conn:
    result = conn.run(service_discovery_cmd, timeout=1800, warn=True, pty=True)

print(result.stdout)
print(result.stderr)
print(result.exited)


## Shared-Prefix Fill Benchmark

This section runs a Parrot-specific microbenchmark based on Parrot's builtin runner and attention kernels. It measures fill latency as shared-prefix length grows and compares Parrot's shared-prefix kernel against Parrot's vLLM-style paged-attention baseline.


In [ ]:
# Shared-prefix fill benchmark configuration.
MODEL_NAME = "lmsys/vicuna-7b-v1.3"
SHARED_PREFIX_LENGTHS = [256, 512, 1024, 2048]
BATCH_SIZE = 8
DIVERGED_LENGTH = 32
REPEATS = 3
NUM_KV_CACHE_BLOCKS = 1024
BLOCK_SIZE = 16
MAX_SEQ_LEN = 4096

ATTN_FUNCS = {
    "parrot_shared_prefix": "xformers_fill_shared_prompts_generate",
    "vllm_paged_attention": "xformers_fill_vllm_paged_attention_generate",
}

print("MODEL_NAME:", MODEL_NAME)
print("SHARED_PREFIX_LENGTHS:", SHARED_PREFIX_LENGTHS)
print("BATCH_SIZE:", BATCH_SIZE)
print("DIVERGED_LENGTH:", DIVERGED_LENGTH)
print("REPEATS:", REPEATS)
print("NUM_KV_CACHE_BLOCKS:", NUM_KV_CACHE_BLOCKS)
print("MAX_SEQ_LEN:", MAX_SEQ_LEN)
print("ATTN_FUNCS:", ATTN_FUNCS)


In [ ]:
from pathlib import Path

compat_and_helpers_sh = r'''#!/usr/bin/env bash
set -euxo pipefail

export CUDA_HOME=/usr/local/cuda-12.6
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:${LD_LIBRARY_PATH:-}"
source /home/cc/parrot-venv/bin/activate

cd /home/cc/ParrotServe
python -m pip install \
  'numpy<2' \
  'setuptools<81' \
  'triton==2.1.0' \
  'transformers==4.31.0' \
  'tokenizers==0.13.3' \
  'huggingface-hub==0.16.4' \
  'sentencepiece==0.1.99'

cd /home/cc/ParrotServe/3rdparty/vllm
python -m pip install -e . --no-build-isolation

cd /home/cc/ParrotServe
python -m pip install -e . --no-build-isolation
'''

run_shared_prefix_benchmark_py = r'''
import argparse
import json
import time

import torch

from parrot.engine.builtin.builtin_runner import BuiltinRunner
from parrot.engine.config import BuiltinConfig
from parrot.engine.primitive_job import Fill


def run_one_measurement(
    runner,
    batch_size: int,
    shared_len: int,
    diverged_len: int,
    base_context_id: int,
):
    shared_fill = Fill(
        session_id=0,
        task_id=0,
        context_id=base_context_id,
        parent_context_id=-1,
        token_ids=[100] * shared_len,
    )

    diverged_fills = []
    for idx in range(batch_size):
        context_id = base_context_id + idx + 1
        diverged_token = 200 + idx
        diverged_fills.append(
            Fill(
                session_id=0,
                task_id=0,
                context_id=context_id,
                parent_context_id=base_context_id,
                token_ids=[diverged_token] * diverged_len,
            )
        )

    total_e2e_ns = 0
    total_model_ns = 0

    e2e_ns, model_ns = runner.run_iter([shared_fill])
    shared_fill_e2e_ns = e2e_ns
    shared_fill_model_ns = model_ns
    total_e2e_ns += e2e_ns
    total_model_ns += model_ns

    e2e_ns, model_ns = runner.run_iter(diverged_fills)
    diverged_fill_e2e_ns = e2e_ns
    diverged_fill_model_ns = model_ns
    total_e2e_ns += e2e_ns
    total_model_ns += model_ns

    for idx in range(batch_size):
        runner.context_manager.free_context(base_context_id + idx + 1)
    runner.context_manager.free_context(base_context_id)

    return {
        "shared_fill_e2e_ms": shared_fill_e2e_ns / 1e6,
        "shared_fill_model_ms": shared_fill_model_ns / 1e6,
        "diverged_fill_e2e_ms": diverged_fill_e2e_ns / 1e6,
        "diverged_fill_model_ms": diverged_fill_model_ns / 1e6,
        "total_fill_e2e_ms": total_e2e_ns / 1e6,
        "total_fill_model_ms": total_model_ns / 1e6,
        "per_request_fill_e2e_ms": (total_e2e_ns / 1e6) / batch_size,
        "per_request_fill_model_ms": (total_model_ns / 1e6) / batch_size,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-name", required=True)
    parser.add_argument("--attn-func", required=True)
    parser.add_argument("--shared-lengths", nargs="+", type=int, required=True)
    parser.add_argument("--batch-size", type=int, required=True)
    parser.add_argument("--diverged-length", type=int, required=True)
    parser.add_argument("--repeats", type=int, required=True)
    parser.add_argument("--num-kv-cache-blocks", type=int, default=1024)
    parser.add_argument("--block-size", type=int, default=16)
    parser.add_argument("--max-seq-len", type=int, default=4096)
    args = parser.parse_args()

    config = BuiltinConfig(
        num_kv_cache_blocks=args.num_kv_cache_blocks,
        attn_func=args.attn_func,
        block_size=args.block_size,
        max_seq_len=args.max_seq_len,
    )

    runner = BuiltinRunner(args.model_name, config=config)
    torch.cuda.synchronize()

    base_context_id = 0
    for shared_len in args.shared_lengths:
        for repeat in range(args.repeats):
            torch.cuda.synchronize()
            started = time.perf_counter()
            metrics = run_one_measurement(
                runner=runner,
                batch_size=args.batch_size,
                shared_len=shared_len,
                diverged_len=args.diverged_length,
                base_context_id=base_context_id,
            )
            torch.cuda.synchronize()
            wall_ms = (time.perf_counter() - started) * 1000.0
            row = {
                "shared_prefix_length": shared_len,
                "repeat": repeat,
                "wall_ms": wall_ms,
                **metrics,
            }
            print(json.dumps(row))
            base_context_id += args.batch_size + 1

    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()
'''

Path("shared_prefix_compat_and_helpers.sh").write_text(compat_and_helpers_sh)
Path("run_shared_prefix_benchmark.py").write_text(run_shared_prefix_benchmark_py)

with my_server.ssh_connection() as conn:
    conn.put("shared_prefix_compat_and_helpers.sh", "/tmp/shared_prefix_compat_and_helpers.sh")
    conn.put("run_shared_prefix_benchmark.py", "/tmp/run_shared_prefix_benchmark.py")
    conn.run("chmod +x /tmp/shared_prefix_compat_and_helpers.sh", warn=True, pty=True)
    result = conn.run("bash /tmp/shared_prefix_compat_and_helpers.sh", timeout=7200, warn=True, pty=True)

print(result.stdout)
print(result.stderr)
print(result.exited)
print(Path("run_shared_prefix_benchmark.py").resolve())
print("Uploaded fill-only benchmark helper files and refreshed the Parrot environment.")


In [ ]:
import csv
import json
from pathlib import Path

rows = []

with my_server.ssh_connection() as conn:
    for system_name, attn_func in ATTN_FUNCS.items():
        cmd = (
            "export CUDA_HOME=/usr/local/cuda-12.6 && "
            "export PATH=$CUDA_HOME/bin:$PATH && "
            "export LD_LIBRARY_PATH=$CUDA_HOME/lib64:${LD_LIBRARY_PATH:-} && "
            "source /home/cc/parrot-venv/bin/activate && "
            "cd /home/cc/ParrotServe && "
            "python /tmp/run_shared_prefix_benchmark.py "
            f"--model-name {MODEL_NAME!r} "
            f"--attn-func {attn_func!r} "
            f"--shared-lengths {' '.join(str(x) for x in SHARED_PREFIX_LENGTHS)} "
            f"--batch-size {BATCH_SIZE} "
            f"--diverged-length {DIVERGED_LENGTH} "
            f"--repeats {REPEATS} "
            f"--num-kv-cache-blocks {NUM_KV_CACHE_BLOCKS} "
            f"--block-size {BLOCK_SIZE} "
            f"--max-seq-len {MAX_SEQ_LEN}"
        )
        result = conn.run(cmd, timeout=7200, warn=True, pty=True)
        print(result.stdout)
        print(result.stderr)
        if result.exited != 0:
            raise RuntimeError(f"Benchmark failed for {system_name}")

        for line in result.stdout.splitlines():
            line = line.strip()
            if not line or not line.startswith("{"):
                continue
            row = json.loads(line)
            row["system"] = system_name
            rows.append(row)

if not rows:
    raise RuntimeError("No benchmark rows were collected.")

local_csv = Path("shared_prefix_fill_benchmark.csv")
with local_csv.open("w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "system",
            "shared_prefix_length",
            "repeat",
            "wall_ms",
            "shared_fill_e2e_ms",
            "shared_fill_model_ms",
            "diverged_fill_e2e_ms",
            "diverged_fill_model_ms",
            "total_fill_e2e_ms",
            "total_fill_model_ms",
            "per_request_fill_e2e_ms",
            "per_request_fill_model_ms",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

print(local_csv.resolve())
print(f"Wrote {len(rows)} rows")


In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

agg = defaultdict(list)
for row in rows:
    agg[(row["system"], row["shared_prefix_length"])].append(row["per_request_fill_e2e_ms"])

summary = []
for (system, shared_prefix_length), vals in sorted(agg.items()):
    summary.append({
        "system": system,
        "shared_prefix_length": shared_prefix_length,
        "mean_per_request_fill_e2e_ms": sum(vals) / len(vals),
        "num_runs": len(vals),
    })

systems = sorted({row["system"] for row in summary})

plt.figure(figsize=(7, 4.5))
for system in systems:
    xs = [row["shared_prefix_length"] for row in summary if row["system"] == system]
    ys = [row["mean_per_request_fill_e2e_ms"] for row in summary if row["system"] == system]
    plt.plot(xs, ys, marker="o", linewidth=2, label=system)

plt.xlabel("Shared prefix length (tokens)")
plt.ylabel("Mean per-request fill latency (ms)")
plt.title("Parrot Shared-Prefix Fill Benchmark")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

summary


In [ ]:
print("Shared-prefix fill benchmark results are saved in shared_prefix_fill_benchmark.csv")


In [ ]:
print("Done")


## Cleanup

When you are done, clean up the server and lease to avoid burning project resources. Leave these cells commented until you are certain you no longer need the instance.


In [ ]:
# Destructive operations: uncomment only when you are done.
# my_server.delete()
# my_lease.delete()
